# SLM Spike Detection GUI

Load the SLM traces, build a stable per-cell bundle, and launch the browser-based tuning GUI.


In [1]:
from pathlib import Path
import subprocess
import sys
import webbrowser

repo_root = Path('/Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline')
figure_dir = repo_root / 'miniVI_PlaceCell_analysis_V4' / 'Figure1_code'
package_parent = repo_root / 'miniVI_PlaceCell_analysis_V4'
if str(package_parent) not in sys.path:
    sys.path.insert(0, str(package_parent))

from dash_slm_detection_app.data_io import build_slm_bundle, save_bundle_pickle

slm_root = Path('/Volumes/QixinT9/AdamLab/Data/SLM_data')
sampling_rate_hz = 500
initial_bad_frames = 50
host = '127.0.0.1'
port = 8052
bundle_output_path = figure_dir / 'SLM_spike_detection_bundle.pkl'
results_output_path = figure_dir / 'SLM_spike_detection_results.pkl'
app_script = package_parent / 'dash_slm_detection_app' / 'app.py'


In [2]:
bundle = build_slm_bundle(
    slm_root,
    sampling_rate_hz=sampling_rate_hz,
    initial_bad_frames=initial_bad_frames,
)
save_bundle_pickle(bundle, bundle_output_path)

for condition_label in ('1x', '20x'):
    print(
        f"{condition_label}: {bundle['traces_by_condition'][condition_label].shape[0]} cells, "
        f"shape={bundle['traces_by_condition'][condition_label].shape}"
    )

print(f"Total cells: {bundle['n_cells_total']}")
print(f"Saved launcher bundle to {bundle_output_path}")
print(f"GUI results will be saved to {results_output_path}")


1x: 11 cells, shape=(11, 300000)
20x: 9 cells, shape=(9, 300000)
Total cells: 20
Saved launcher bundle to /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/Figure1_code/SLM_spike_detection_bundle.pkl
GUI results will be saved to /Users/qixinyang/Documents/AdamLab/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/Figure1_code/SLM_spike_detection_results.pkl


In [3]:
if not app_script.is_file():
    raise FileNotFoundError(f'Cannot find app.py at: {app_script}')

browser_host = '127.0.0.1' if host in ('0.0.0.0', '::', '') else host
url = f'http://{browser_host}:{int(port)}'

if '_slm_gui_proc' in globals() and _slm_gui_proc is not None and _slm_gui_proc.poll() is None:
    _slm_gui_proc.terminate()

cmd = [
    sys.executable,
    str(app_script),
    str(bundle_output_path),
    '--host', str(host),
    '--port', str(int(port)),
    '--results-path', str(results_output_path),
    '--no-browser',
]

_slm_gui_proc = subprocess.Popen(
    cmd,
    cwd=str(package_parent),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)

_ = webbrowser.open(url)
print(f'Launched SLM GUI at {url}')


Launched SLM GUI at http://127.0.0.1:8052
